# Laboratorio L10 - Addestrare e valutare un modello

B.6 - Data science e Machine Learning: dai dati ai modelli

File necessario: `pinguini.csv`.

Dati: Palmer Penguins, Horst AM, Hill AP, Gorman KB (2020), licenza CC0, https://allisonhorst.github.io/palmerpenguins/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def controlla(condizione, suggerimento="Non ancora: rileggere la consegna."):
    """Stampa un riscontro immediato sull'esercizio."""
    print("Corretto." if condizione else suggerimento)

pinguini = pd.read_csv("pinguini.csv").dropna(subset=["becco_lunghezza_mm", "pinna_lunghezza_mm", "massa_g"])
X = pinguini[["becco_lunghezza_mm", "pinna_lunghezza_mm"]]
y = pinguini["specie"]
print(len(pinguini), "pinguini")

## 1. Dividere i dati (esercizio 1)

`train_test_split` divide caratteristiche ed etichette in due parti:

- `test_size=0.25`: il 25% degli esempi va nell'insieme di verifica (test), il 75% in quello di addestramento
- `random_state=42`: la divisione è casuale; il seme la rende ripetibile (con lo stesso seme si ottiene la stessa divisione)
- `stratify=y`: mantiene in entrambe le parti le stesse proporzioni delle specie

Restituisce quattro oggetti, nell'ordine: caratteristiche di addestramento, caratteristiche di verifica, etichette di addestramento, etichette di verifica.

In [ ]:
from sklearn.model_selection import train_test_split

X_add, X_ver, y_add, y_ver = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("addestramento:", len(X_add), " verifica:", len(X_ver))
print(y_ver.value_counts())
controlla(len(X_ver) == 86)

## 2. La baseline (esercizio 1)

Un modello "stupido" che prevede sempre la classe più frequente nei dati di addestramento. Ogni modello vero deve fare meglio.

`DummyClassifier(strategy="most_frequent")` realizza questa baseline con la stessa interfaccia degli altri modelli.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_add, y_add)
print("la baseline prevede sempre:", baseline.predict(X_ver[:1])[0])
print("accuratezza della baseline sulla verifica:", round(baseline.score(X_ver, y_ver), 3))

## 3. Addestrare su una parte, valutare sull'altra (esercizio 2)

Regola: `fit` solo sui dati di addestramento; `score` e `predict` sui dati di verifica.

Per il k-NN la standardizzazione si calcola sui soli dati di addestramento (`fit`) e si applica a entrambe le parti (`transform`): i dati di verifica devono restare "mai visti" in ogni fase.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

scala = StandardScaler().fit(X_add)
risultati = []
for k in [1, 5, 15]:
    m = KNeighborsClassifier(n_neighbors=k).fit(scala.transform(X_add), y_add)
    risultati.append((f"k-NN k={k}", m.score(scala.transform(X_add), y_add), m.score(scala.transform(X_ver), y_ver)))
for p in [1, 2, 3, None]:
    a = DecisionTreeClassifier(max_depth=p, random_state=0).fit(X_add, y_add)
    risultati.append((f"albero prof.={p}", a.score(X_add, y_add), a.score(X_ver, y_ver)))

tabella = pd.DataFrame(risultati, columns=["modello", "acc. addestramento", "acc. verifica"]).round(3)
tabella

**Domande**

- Quali modelli hanno accuratezza 1,0 sull'addestramento? Anche sulla verifica?
- Quale modello scegliereste? In base a quale colonna?

Risposte:

## 4. Matrice di confusione (esercizio 3)

- `confusion_matrix(veri, previsti, labels=...)`: righe = classe vera, colonne = classe prevista
- `ConfusionMatrixDisplay`: disegna la matrice

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

albero = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_add, y_add)
previsti = albero.predict(X_ver)
SPECIE = ["Adelie", "Chinstrap", "Gentoo"]
cm = confusion_matrix(y_ver, previsti, labels=SPECIE)
pd.DataFrame(cm, index=["vera " + s for s in SPECIE], columns=["prevista " + s for s in SPECIE])

In [ ]:
ConfusionMatrixDisplay(cm, display_labels=SPECIE).plot(cmap="Blues", colorbar=False)
plt.show()

Completare: l'accuratezza è la somma della diagonale divisa per il totale. `np.trace(cm)` somma la diagonale, `cm.sum()` somma tutti i valori.

In [ ]:
acc = ___ / ___
print(round(acc, 3))
controlla(abs(acc - albero.score(X_ver, y_ver)) < 1e-9)

## 5. Classi sbilanciate: precisione e richiamo (esercizio 4)

Nuovo problema: riconoscere i Chinstrap (circa il 20% dei pinguini). Le etichette diventano due: `Chinstrap` e `altro`.

- precisione: tra i pinguini previsti Chinstrap, quanti lo sono davvero
- richiamo (recall, sensibilità): tra i Chinstrap veri, quanti vengono riconosciuti
- `pos_label="Chinstrap"`: indica la classe "positiva", quella da riconoscere

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

y2 = np.where(y == "Chinstrap", "Chinstrap", "altro")
X_add2, X_ver2, y_add2, y_ver2 = train_test_split(X, y2, test_size=0.25, random_state=42, stratify=y2)

def valuta(nome, previsti):
    print(f"{nome:10} accuratezza {accuracy_score(y_ver2, previsti):.3f}  "
          f"precisione {precision_score(y_ver2, previsti, pos_label='Chinstrap', zero_division=0):.3f}  "
          f"richiamo {recall_score(y_ver2, previsti, pos_label='Chinstrap'):.3f}")

base2 = DummyClassifier(strategy="most_frequent").fit(X_add2, y_add2)
valuta("baseline", base2.predict(X_ver2))
alb2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_add2, y_add2)
valuta("albero", alb2.predict(X_ver2))
pd.DataFrame(confusion_matrix(y_ver2, alb2.predict(X_ver2), labels=["Chinstrap", "altro"]),
             index=["vero Chinstrap", "vero altro"], columns=["previsto Chinstrap", "previsto altro"])

**Domande**

- La baseline ha accuratezza 0,80 e non riconosce nessun Chinstrap. Che cosa insegna sull'accuratezza con classi sbilanciate?
- L'albero ha precisione 1 e richiamo basso: che cosa significa, in parole?
- Provare `max_depth=3` e `max_depth=4` nella cella seguente: come cambiano precisione e richiamo?

Risposte:

In [ ]:
for p in [2, 3, 4]:
    a = DecisionTreeClassifier(max_depth=___, random_state=0).fit(X_add2, y_add2)
    valuta(f"prof. {p}", a.predict(X_ver2))

## 6. Approfondimento (esercizio 5)

`classification_report` riassume precisione, richiamo e F1 (una media delle due) per ogni classe. Applicarlo all'albero del punto 4, sulle tre specie.

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_ver, previsti, digits=3))